# 11 — Finetune on CFD Validation Error (Module 7 — Track 1 Close-Out)

Loads the 3 CFD validation HDF5 cases from `10_cfd_validation.ipynb`,
then finetunes the **MultiHeadMGN (MeshGraphNet)** surrogate heads on the
prediction errors to close the learning loop.

| Step | What | Cell |
|------|------|------|
| 1 | Load validation HDF5 + model | cell-load |
| 2 | Baseline metrics (before finetune) | cell-baseline |
| 3 | Finetune loop (Adam lr=1e-5, 100 epochs) | cell-finetune |
| 4 | Before vs after comparison | cell-eval |
| 5 | Save `multihead_finetuned.pt` | cell-save |

**Prerequisite:** `10_cfd_validation.ipynb` Cell 5 must have completed,
writing 3 HDF5 files to `checkpoints/cfd_validation/hdf5/`.

In [ ]:
# ── Cell 0: Setup ──────────────────────────────────────────────────────────
import os, subprocess, sys

if not os.path.exists('/content/cfd-ald-app'):
    subprocess.run(['git', 'clone', 'https://github.com/Ranaam21/cfd-ald-app.git',
                    '/content/cfd-ald-app'], check=True)
else:
    subprocess.run(['git', '-C', '/content/cfd-ald-app', 'pull'], check=False)

sys.path.insert(0, '/content/cfd-ald-app')
print('Repo contents:', os.listdir('/content/cfd-ald-app'))

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'h5py', 'torch_geometric'], check=False)

import copy, json
import numpy as np
import h5py
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.spatial import cKDTree

DRIVE_BASE   = Path('/content/drive/MyDrive/cfd-ald-app')
HDF5_VAL     = DRIVE_BASE / 'checkpoints' / 'cfd_validation' / 'hdf5'
CKPT_IN      = DRIVE_BASE / 'checkpoints' / 'multihead' / 'multihead_final.pt'
CKPT_OUT     = DRIVE_BASE / 'checkpoints' / 'multihead' / 'multihead_finetuned.pt'
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

GLOBAL_KEYS = [
    'Re', 'Pr', 'Sc', 'Ma', 'Pe_h', 'Pe_m', 'Da',
    'D', 'pitch_over_D', 'H_plenum', 't_face', 'standoff',
    'flow_rate_slm', 'beta', 'v_th', 'D_m', 'n_holes', 'open_area_frac',
]
FIELD_NAMES = ['Ux', 'Uy', 'Uz', 'p', 'T [K]', 'TMA']

print(f'Device: {DEVICE}')
print(f'HDF5 val dir: {HDF5_VAL}')

In [ ]:
# ── Cell 1: Load validation HDF5 cases + MultiHeadMGN surrogate ────────────
def mlp(in_dim, out_dim, hidden=None, n_layers=2):
    hidden = hidden or out_dim
    dims   = [in_dim] + [hidden] * (n_layers - 1) + [out_dim]
    mods   = []
    for i in range(len(dims) - 1):
        mods.append(nn.Linear(dims[i], dims[i+1]))
        if i < len(dims) - 2:
            mods.append(nn.SiLU())
    return nn.Sequential(*mods)

from torch_geometric.nn import MessagePassing

class MGNProcessor(MessagePassing):
    def __init__(self, hidden):
        super().__init__(aggr='sum')
        self.edge_mlp  = mlp(3*hidden, hidden)
        self.node_mlp  = mlp(2*hidden, hidden)
        self.edge_norm = nn.LayerNorm(hidden)
        self.node_norm = nn.LayerNorm(hidden)
    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        new_e = self.edge_norm(self.edge_mlp(torch.cat([x[src], x[dst], edge_attr], -1)) + edge_attr)
        agg   = self.propagate(edge_index, x=x, edge_attr=new_e, size=(x.size(0), x.size(0)))
        new_x = self.node_norm(self.node_mlp(torch.cat([x, agg], -1)) + x)
        return new_x, new_e
    def message(self, edge_attr): return edge_attr

class MultiHeadMGN(nn.Module):
    def __init__(self, node_dim, edge_dim, flow_out=4, heat_out=1,
                 species_out=1, hidden=128, n_layers=8):
        super().__init__()
        self.node_enc    = mlp(node_dim, hidden)
        self.edge_enc    = mlp(edge_dim, hidden)
        self.processors  = nn.ModuleList([MGNProcessor(hidden) for _ in range(n_layers)])
        self.flow_dec    = mlp(hidden, flow_out)
        self.heat_dec    = mlp(hidden, heat_out)
        self.species_dec = mlp(hidden, species_out)
    def forward(self, x, edge_index, edge_attr):
        x = self.node_enc(x)
        e = self.edge_enc(edge_attr)
        for proc in self.processors:
            x, e = proc(x, edge_index, e)
        return self.flow_dec(x), self.heat_dec(x), self.species_dec(x)

# Load checkpoint
ckpt       = torch.load(CKPT_IN, map_location='cpu')
cfg        = ckpt['cfg']
norm       = ckpt['norm']
node_mean  = np.array(norm['node_mean'], dtype=np.float32)
node_std   = np.array(norm['node_std'],  dtype=np.float32)
out_mean   = np.array(norm['out_mean'],  dtype=np.float32)
out_std    = np.array(norm['out_std'],   dtype=np.float32)
K          = cfg['k_neighbors']

model = MultiHeadMGN(
    node_dim=cfg['node_input_dim'], edge_dim=cfg['edge_input_dim'],
    flow_out=cfg['flow_out_dim'],   heat_out=cfg['heat_out_dim'],
    species_out=cfg['species_out_dim'],
    hidden=cfg['hidden_dim'],       n_layers=cfg['n_layers'],
).to(DEVICE)
model.load_state_dict(ckpt['model'])
print(f'Model: {sum(p.numel() for p in model.parameters()):,} parameters')

# Load validation HDF5 cases
val_h5_files = sorted(HDF5_VAL.glob('cfd_val_*.h5'))
assert val_h5_files, f'No cfd_val_*.h5 found in {HDF5_VAL} — run 10_cfd_validation.ipynb first'
print(f'Found {len(val_h5_files)} validation HDF5 files:')
for p in val_h5_files:
    print(f'  {p.name}')

def load_h5_case(h5_path):
    """Load one HDF5 case → (coords, node_inp, edge_index, edge_attr, truth_norm)."""
    with h5py.File(h5_path, 'r') as f:
        coords = f['coords'][:].astype(np.float32)
        nf     = f['inputs/node_features'][:].astype(np.float32)   # [N, 4]
        gf     = f['inputs/global'][:].astype(np.float32)           # [18]
        truth  = f['outputs/node_fields'][:].astype(np.float32)     # [N, 6]

    N  = len(coords)
    xi = np.concatenate([nf, np.tile(gf, (N, 1))], axis=1)          # [N, 22]

    # k-NN graph
    tree = cKDTree(coords)
    _, idx = tree.query(coords, k=K+1); idx = idx[:, 1:]
    src  = np.repeat(np.arange(N), K); dst = idx.flatten()
    diff = coords[dst] - coords[src]
    dist = np.linalg.norm(diff, axis=1, keepdims=True)
    med  = float(np.median(dist)) + 1e-8
    ef   = np.concatenate([diff/med, dist/med], axis=1).astype(np.float32)  # [N*K, 4]

    # Normalise inputs
    xi_n = (xi - node_mean) / node_std

    # Normalise targets
    truth_n = np.zeros_like(truth)
    truth_n[:, :4] = (truth[:, :4] - out_mean[:4]) / (out_std[:4] + 1e-8)
    truth_n[:, 4]  = (truth[:, 4]  - out_mean[4])  / (out_std[4]  + 1e-8)
    truth_n[:, 5]  = (truth[:, 5]  - out_mean[5])  / (out_std[5]  + 1e-8)

    return {
        'name':    h5_path.stem,
        'x':       torch.from_numpy(xi_n),
        'ei':      torch.tensor(np.stack([src, dst]), dtype=torch.long),
        'ea':      torch.from_numpy(ef),
        'truth_n': torch.from_numpy(truth_n),
        'truth':   truth,   # denormalised ground truth (for metrics)
    }

print('\nBuilding k-NN graphs (may take ~1 min per case on CPU)…')
val_cases = [load_h5_case(p) for p in val_h5_files]
print('Graphs ready.')

In [ ]:
# ── Cell 2: Baseline metrics (before finetune) ─────────────────────────────
def evaluate(model, cases):
    """Return per-field RMSE (denormalised) for all cases combined."""
    model.eval()
    all_preds = []
    all_truth = []
    with torch.no_grad():
        for c in cases:
            x  = c['x'].to(DEVICE)
            ei = c['ei'].to(DEVICE)
            ea = c['ea'].to(DEVICE)
            fp, hp, sp = model(x, ei, ea)
            fp = fp.cpu().numpy(); hp = hp.cpu().numpy().flatten(); sp = sp.cpu().numpy().flatten()
            pred = np.zeros((len(fp), 6), dtype=np.float32)
            pred[:, :4] = fp * out_std[:4] + out_mean[:4]
            pred[:, 4]  = hp * out_std[4]  + out_mean[4]
            pred[:, 5]  = sp * out_std[5]  + out_mean[5]
            all_preds.append(pred)
            all_truth.append(c['truth'])
    all_preds = np.concatenate(all_preds, axis=0)
    all_truth = np.concatenate(all_truth, axis=0)
    rmse = np.sqrt(np.mean((all_preds - all_truth)**2, axis=0))
    return dict(zip(FIELD_NAMES, rmse.tolist()))

print('Baseline (before finetune):')
baseline_rmse = evaluate(model, val_cases)
print(f'  {"Field":<10} {"RMSE"}')
print(f'  {"-"*10} {"------"}')
for k, v in baseline_rmse.items():
    print(f'  {k:<10} {v:.6g}')

In [ ]:
# ── Cell 3: Finetune loop ──────────────────────────────────────────────────
# Strategy: all 3 validation cases as training data.
# Very small LR (1e-5) + weight decay to avoid catastrophic forgetting.
# 100 epochs; save best combined RMSE checkpoint.

FINETUNE_LR     = 1e-5
FINETUNE_WD     = 1e-6
FINETUNE_EPOCHS = 100
LOSS_WEIGHTS    = {'flow': 1.0, 'heat': 2.0, 'species': 2.0}  # upweight ALD-specific heads

# Work on a copy so baseline_rmse stays comparable
model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=FINETUNE_LR, weight_decay=FINETUNE_WD)
mse_loss  = nn.MSELoss()

best_loss   = float('inf')
best_state  = copy.deepcopy(model.state_dict())
train_losses = []

for epoch in range(1, FINETUNE_EPOCHS + 1):
    epoch_loss = 0.0
    for c in val_cases:
        x       = c['x'].to(DEVICE)
        ei      = c['ei'].to(DEVICE)
        ea      = c['ea'].to(DEVICE)
        truth_n = c['truth_n'].to(DEVICE)  # normalised targets

        optimizer.zero_grad()
        fp, hp, sp = model(x, ei, ea)

        loss = (
            LOSS_WEIGHTS['flow']    * mse_loss(fp, truth_n[:, :4]) +
            LOSS_WEIGHTS['heat']    * mse_loss(hp.squeeze(-1), truth_n[:, 4]) +
            LOSS_WEIGHTS['species'] * mse_loss(sp.squeeze(-1), truth_n[:, 5])
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(val_cases)
    train_losses.append(avg_loss)

    if avg_loss < best_loss:
        best_loss  = avg_loss
        best_state = copy.deepcopy(model.state_dict())

    if epoch % 10 == 0 or epoch == 1:
        print(f'  Epoch {epoch:3d}/{FINETUNE_EPOCHS}  loss={avg_loss:.6f}  best={best_loss:.6f}')

# Restore best
model.load_state_dict(best_state)
model.eval()

# Loss curve
plt.figure(figsize=(8, 3))
plt.plot(train_losses)
plt.xlabel('Epoch'); plt.ylabel('Weighted MSE loss')
plt.title('Finetune loss curve')
plt.tight_layout(); plt.show()

print(f'\nFinetune complete. Best epoch loss: {best_loss:.6f}')

In [ ]:
# ── Cell 4: Before vs after comparison ────────────────────────────────────
after_rmse = evaluate(model, val_cases)

print(f'  {"Field":<10} {"RMSE before":>14} {"RMSE after":>12} {"Δ %":>10}')
print(f'  {"-"*10} {"-"*14} {"-"*12} {"-"*10}')
for fname in FIELD_NAMES:
    b = baseline_rmse[fname]
    a = after_rmse[fname]
    delta_pct = (a - b) / (b + 1e-12) * 100
    arrow = '↓' if delta_pct < 0 else '↑'
    print(f'  {fname:<10} {b:>14.6g} {a:>12.6g} {arrow} {abs(delta_pct):>7.2f}%')

# Field slice: before vs after vs CFD truth for T and TMA on first case
c0 = val_cases[0]
with h5py.File(val_h5_files[0], 'r') as f:
    coords = f['coords'][:].astype(np.float32)
    truth  = f['outputs/node_fields'][:].astype(np.float32)

# Load original model for 'before' predictions
model_before = MultiHeadMGN(
    node_dim=cfg['node_input_dim'], edge_dim=cfg['edge_input_dim'],
    flow_out=cfg['flow_out_dim'],   heat_out=cfg['heat_out_dim'],
    species_out=cfg['species_out_dim'],
    hidden=cfg['hidden_dim'],       n_layers=cfg['n_layers'],
).to(DEVICE)
model_before.load_state_dict(torch.load(CKPT_IN, map_location='cpu')['model'])
model_before.eval()

def get_preds(m, c):
    with torch.no_grad():
        x = c['x'].to(DEVICE); ei = c['ei'].to(DEVICE); ea = c['ea'].to(DEVICE)
        fp, hp, sp = m(x, ei, ea)
    fp = fp.cpu().numpy(); hp = hp.cpu().numpy().flatten(); sp = sp.cpu().numpy().flatten()
    pred = np.zeros((len(fp), 6), dtype=np.float32)
    pred[:, :4] = fp * out_std[:4] + out_mean[:4]
    pred[:, 4]  = hp * out_std[4]  + out_mean[4]
    pred[:, 5]  = sp * out_std[5]  + out_mean[5]
    return pred

preds_before = get_preds(model_before, c0)
preds_after  = get_preds(model,        c0)

z     = coords[:, 2]
z_th  = z.min() + 0.15 * (z.max() - z.min())
mask  = z < z_th
rng   = np.random.default_rng(0)
sel   = rng.choice(mask.sum(), min(3000, mask.sum()), replace=False)
xp, yp = coords[mask][sel, 0]*1e3, coords[mask][sel, 1]*1e3

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for row, (fi, fname) in enumerate([(4, 'T [K]'), (5, 'TMA')]):
    for col, (vals, label) in enumerate([
        (truth,         'CFD truth'),
        (preds_before,  'Surrogate (before)'),
        (preds_after,   'Surrogate (after)'),
    ]):
        fv = vals[mask][sel, fi]
        vmin, vmax = truth[mask][sel, fi].min(), truth[mask][sel, fi].max()
        sc = axes[row, col].scatter(xp, yp, c=fv, s=1, cmap='turbo', vmin=vmin, vmax=vmax)
        plt.colorbar(sc, ax=axes[row, col], pad=0.02)
        axes[row, col].set_title(f'{label} — {fname}', fontsize=9)
        axes[row, col].set_aspect('equal'); axes[row, col].axis('off')
fig.suptitle(f'Field slices — {val_h5_files[0].stem} (bottom 15% plenum)')
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 5: Save multihead_finetuned.pt ────────────────────────────────────
# Same schema as multihead_final.pt — app.py can load either without changes.
torch.save({
    'model': model.state_dict(),
    'cfg':   cfg,
    'norm':  norm,
    'finetune_meta': {
        'base_ckpt':    str(CKPT_IN),
        'n_val_cases':  len(val_cases),
        'epochs':       FINETUNE_EPOCHS,
        'lr':           FINETUNE_LR,
        'loss_weights': LOSS_WEIGHTS,
        'best_loss':    best_loss,
        'baseline_rmse': baseline_rmse,
        'finetuned_rmse': after_rmse,
    },
}, CKPT_OUT)
print(f'Saved → {CKPT_OUT}')

# Improvement summary
improved = sum(1 for k in FIELD_NAMES if after_rmse[k] < baseline_rmse[k])
print(f'\n{improved}/{len(FIELD_NAMES)} fields improved after finetuning.')
print()
print('Track 1 (PCGM — Physics-Constrained Geometric Morphogenesis) complete:')
print('  Module 0  Guardrails          ✓')
print('  Module 1  Geometry (PCGM)     ✓')
print('  Module 2  Data Layer          ✓')
print('  Module 3  Data Standardisation✓')
print('  Module 4  ML Surrogate        ✓')
print('  Module 5  Guardrail Engine    ✓')
print('  Module 6  Optimizer           ✓')
print('  Module 7  Verification Loop   ✓  ← multihead_finetuned.pt')
print()
print('Next: Track 2 — VICES (Voxel-Implicit Computational Engineering Synthesis)')